In [1]:
#next word prediction

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv('1661-0.txt',sep='\t',header=None,names=["Texts"])

In [6]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['Texts'])

In [7]:
voc_size = len(tokenizer.index_word)
voc_size

8930

In [8]:
tokenized_sentence = tokenizer.texts_to_sequences(df['Texts'])
tokenized_sentence[0]

[145, 4789, 1, 1020, 4, 128, 34, 45, 611, 2235, 2236]

In [9]:
input_sequence = []

for j in range(0,len(tokenized_sentence)):
    for i in range(1,len(tokenized_sentence[j])):
        input_sequence.append(tokenized_sentence[j][:i+1])

In [10]:
max_len = max([len(x) for x in input_sequence])
max_len

20

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequence = pad_sequences(input_sequence, maxlen = max_len, padding='pre')

In [12]:
x= padded_input_sequence[:,:-1]
y = padded_input_sequence[:,-1]

In [13]:
x[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0, 145], dtype=int32)

In [14]:
input_shape = len(x[0])
input_shape

19

In [15]:
len(y)

101619

In [16]:
y = to_categorical(y,num_classes=voc_size+1)


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,Embedding

model = Sequential()
model.add(Embedding(voc_size+1, 100, input_shape=(input_shape,)))
model.add(LSTM(150))
model.add(Dense(voc_size+1, activation='softmax'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 19, 100)             │         893,100 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 150)                 │         150,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 8931)                │       1,348,581 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,392,281 (9.13 MB)

 Trainable params: 2,392,281 (9.13 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# prompt: data visualization

import matplotlib.pyplot as plt

# Assuming 'history' object contains the training history from model.fit()
# Example: history = model.fit(x, y, epochs=50)

# Plot training & validation accuracy values
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
history = model.fit(x,y,
    epochs=50,
    batch_size = 32
)

Epoch 1/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 214s 67ms/step - accuracy: 0.0599 - loss: 6.5889
Epoch 2/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 201s 63ms/step - accuracy: 0.1230 - loss: 5.5428
Epoch 3/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 207s 65ms/step - accuracy: 0.1537 - loss: 5.0827
Epoch 4/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 264s 66ms/step - accuracy: 0.1758 - loss: 4.6995
Epoch 5/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 253s 63ms/step - accuracy: 0.1937 - loss: 4.3802
Epoch 6/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 198s 62ms/step - accuracy: 0.2209 - loss: 4.0533
Epoch 7/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 196s 60ms/step - accuracy: 0.2509 - loss: 3.7695
Epoch 8/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 202s 60ms/step - accuracy: 0.2859 - loss: 3.4927
Epoch 9/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 184s 58ms/step - accuracy: 0.3211 - loss: 3.2418
Epoch 10/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 183s 58ms/step - accuracy: 0.3654 - loss: 2.9935
Epoch 11/50
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 204s 58ms/step - accuracy: 0.4008 - loss: 2.77

In [ ]:
import pickle
filename = 'nextWordModel.pkl'
pickle.dump(model, open(filename, 'wb'))

In [ ]:
pickle.dump(tokenizer, open("tokenizer.pkl", "wb"))

In [ ]:
loaded_model = pickle.load(open("/kaggle/working/nextWordModel.pkl", "rb"))
loaded_tokenizer = pickle.load(open("/kaggle/working/tokenizer.pkl", "rb"))

In [ ]:
text = "kill"

for  i in range(18):
    token_text = loaded_tokenizer.texts_to_sequences([text])[0]
    padded_token_text = pad_sequences([token_text],maxlen=19,padding='pre')
    pos = np.argmax(loaded_model.predict(padded_token_text))
    for word,index in loaded_tokenizer.word_index.items():
        if index == pos:
                text = text + " " + word
                print(text)